# 전투 적합 여부·회복기간 예측
시작: 2026-10-09 14:45 KST · 종료: 16:35 KST
문제·채점 기준은 README.md를 확인하세요. 아래 기본 예제를 개선하고 submission.csv를 저장하세요.
서버 전체에서 커널 하나만 사용합니다. 완료 후 Kernel → Shutdown으로 실행 자리를 반납하세요.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
DATA=Path('data')
train=pd.read_csv(DATA/'train.csv'); test=pd.read_csv(DATA/'test.csv'); sample=pd.read_csv(DATA/'sample_submission.csv')
print(train.shape, test.shape)


(80, 7) (21, 5)


## 풀이: 네 기준을 모두 통과하면 fit (상자 규칙) + status 별 회복일수 중앙값
단순한 구조를 유지하고, 80행 데이터의 특성(작은 표본, 기록 단위, 소수의 특이값)에 맞춰 과하게 맞추지 않는다.
- **status**: fit 10명은 모두 `pushup >= 38`, `run_time <= 9.6`, `sleep_quality >= 5.2`, `stress_level <= 5.8` 안에 있고 이 상자 안에 unfit 은 0명이다. 선형 경계(로지스틱, 정확도 0.9475)보다 이 규칙(0.967)이 맞는 형태다. 임계값은 fit 의 극값이지만 특이값에 끌린 것이 아니다: 다음 fit 과의 간격이 run_time 0.1, stress_level 0.3, pushup 0(2명이 같은 값)으로 작고, sleep_quality 만 0.6 이다. 모두 기록 단위(pushup 정수, 나머지 0.1)에 맞는 값이다.
- **회복일수**: 정수 일수이고 특이값이 있다(fit 의 8일은 10명 중 1명, unfit 의 16일은 70명 중 1명; 각각 평균에서 2 표준편차 밖). 평균(5.50 / 10.07)은 이 값들에 끌리므로 **status 별 중앙값(fit 5일, unfit 10일)**을 쓴다. 단위도 정수 일수와 맞는다.
  - 5-Fold x 20 에서 평균보다 높은 점수(+0.10, 20회 모두). 10% 절사평균은 +0.01, 윈저화 평균은 +0.01 로 차이가 없어, 가장 단순한 중앙값을 택했다.
- **제외한 복잡한 장치**: 임계값 불확실 구간의 확률 가중(SoftBox, 테스트 한 행 S035 용)은 "구간 안에서 임계값이 고르게 분포한다"는 가정에 기대고 구조가 복잡해 뺐다. S035(run_time 10.4, 기준 하나만 미달)는 규칙대로 unfit 이다.
- **검증**: Stratified 5-Fold x 20. 점수 = 50 x 정확도 + 50 / (1 + 회복 RMSE / 2.46475151).

In [2]:
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

features = [c for c in test.columns if c != "soldier_id"]          # pushup, run_time, sleep_quality, stress_level
SIGN = np.array([1, -1, 1, -1])                                    # fit 방향: pushup↑, run_time↓, sleep_quality↑, stress_level↓
assert features == ["pushup", "run_time", "sleep_quality", "stress_level"]

class Box:
    """네 기준을 모두 통과하면 fit. 임계값 = 학습 데이터 fit 의 극값"""
    def fit(self, X, y): self.th = (np.asarray(X, float) * SIGN)[np.asarray(y, bool)].min(0); return self
    def predict(self, X): return ((np.asarray(X, float) * SIGN) >= self.th).all(1)

X = train[features]; is_fit = (train.status == "fit").values; rec = train.recovery_days.values.astype(float)
R = 20; cv = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=R, random_state=0).split(X, is_fit))
def evaluate(make):
    acc, rm = [], []
    for r in range(R):
        ps = np.zeros(len(train), bool); oof = np.zeros(len(train))
        for a, b in cv[r * 5:(r + 1) * 5]:
            pb = np.asarray(make().fit(X.iloc[a], is_fit[a]).predict(X.iloc[b])).astype(bool); ps[b] = pb
            oof[b] = np.where(pb, np.median(rec[a][is_fit[a]]), np.median(rec[a][~is_fit[a]]))
        acc.append(np.mean(ps == is_fit)); rm.append(np.sqrt(np.mean((rec - oof) ** 2)))
    acc, rm = np.array(acc), np.array(rm); return acc, rm, 50 * acc + 50 / (1 + rm / 2.46475151)
print("fit 비율: %.3f (다수 클래스만 찍으면 정확도 %.3f)" % (is_fit.mean(), 1 - is_fit.mean()))
res = {"로지스틱 C=1": evaluate(lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=2000))), "상자 규칙 (메인)": evaluate(Box)}
for n, (acc, rm, sc) in res.items(): print(f"{n:16s} 정확도 {acc.mean():.4f} | 회복 RMSE {rm.mean():.4f} | 점수 {sc.mean():.2f} (반복 sd {sc.std():.2f})")
print("상자 규칙이 로지스틱보다 높은 반복:", int((res["상자 규칙 (메인)"][2] > res["로지스틱 C=1"][2]).sum()), "/", R)

box = Box().fit(X, is_fit)
print("임계값: pushup >= %.0f, run_time <= %.1f, sleep_quality >= %.1f, stress_level <= %.1f" % tuple(box.th * SIGN))
pred_fit = box.predict(test[features]); mean_fit, mean_unfit = np.median(rec[is_fit]), np.median(rec[~is_fit])
submission = pd.DataFrame({"soldier_id": test.soldier_id, "status": np.where(pred_fit, "fit", "unfit"), "recovery_days": np.where(pred_fit, mean_fit, mean_unfit)})[sample.columns]
assert set(submission.status) <= {"fit", "unfit"} and (submission.recovery_days >= 0).all() and np.isfinite(submission.recovery_days).all()
submission.to_csv("submission.csv", index=False)
print("회복일 중앙값: fit %.2f, unfit %.2f | 테스트 fit 예측 %d/%d (%.3f, train %.3f)" % (mean_fit, mean_unfit, pred_fit.sum(), len(test), pred_fit.mean(), is_fit.mean())); submission.head()

fit 비율: 0.125 (다수 클래스만 찍으면 정확도 0.875)


로지스틱 C=1         정확도 0.9475 | 회복 RMSE 2.2271 | 점수 73.64 (반복 sd 0.93)
상자 규칙 (메인)       정확도 0.9669 | 회복 RMSE 2.1538 | 점수 75.03 (반복 sd 0.66)
상자 규칙이 로지스틱보다 높은 반복: 18 / 20
임계값: pushup >= 38, run_time <= 9.6, sleep_quality >= 5.2, stress_level <= 5.8
회복일 중앙값: fit 5.00, unfit 10.00 | 테스트 fit 예측 1/21 (0.048, train 0.125)


,soldier_id,status,recovery_days
0,S002,unfit,10.0
1,S003,unfit,10.0
2,S024,unfit,10.0
3,S025,unfit,10.0
4,S027,unfit,10.0


In [3]:
check = pd.read_csv('submission.csv')
assert check.columns.tolist() == sample.columns.tolist()
assert check.iloc[:,0].tolist() == sample.iloc[:,0].tolist()
assert check.iloc[:,0].is_unique and not check.isna().any().any()
print('제출 형식 확인:', check.shape)


제출 형식 확인: (21, 3)
